# Idade dos chamados pendentes

## Objetivo
Calcular uma mediana por fotografia diária, mantendo volume, elegibilidade e desconhecidos visíveis. SQL e Python conferem em 710 resultados (355 dias e dois cenários).

Fonte: [UCI, Amaral et al., 2018](https://doi.org/10.24432/C57S4H), CC BY 4.0. Intervalo de fotografias: 29/02/2016 a 17/02/2017, sem fuso conhecido.

## Preparação
Execute a partir da raiz do repositório ou da pasta notebooks. Requer pandas e os CSVs da etapa 06. Esta conferência usa a saída SQL salva em docs/idade-fila-sql.json; não abre conexão. Consulta reproduzível: sql/05_idade_fila_diaria.sql.

In [1]:
from pathlib import Path
import runpy
import pandas as pd
raiz = Path.cwd()
if not (raiz / 'scripts').exists():
    raiz = raiz.parent
analise = runpy.run_path(str(raiz / 'scripts/10_idade_fila_diaria.py'))

## Etapas
### 1. Recalcular a idade por dia
A idade parte da abertura original e inclui esperas. Unknown não entra como pendente. Um pendente sem idade válida continua contado na fila.

In [2]:
resultados = analise['calcular']()
tabela = pd.DataFrame(resultados)
todos = tabela.loc[tabela.cenario.eq('todos')].copy()
print(todos.loc[todos.data.isin(['2016-02-29', '2016-03-16', '2016-12-15', '2017-02-17'])].to_string(index=False))

      data cenario  n_fila  n_elegivel  n_idade_invalida  n_desconhecido  mediana_idade_horas
2016-02-29   todos     163         163                 0               0            11.133333
2016-03-16   todos    1869        1869                 0               0           157.450000
2016-12-15   todos      33          33                 0               0          5365.766667
2017-02-17   todos       0           0                 0               0                  NaN


### 2. Preservar datas vazias
Fila zero significa ausência de pendentes observados naquele corte. A idade mediana é ausente; não é duração zero e não comprova que toda a operação real tenha ficado sem chamados.

In [3]:
print(todos.loc[todos.n_fila.eq(0), ['data', 'n_fila', 'mediana_idade_horas']].to_string(index=False))

      data  n_fila  mediana_idade_horas
2017-02-14       0                  NaN
2017-02-17       0                  NaN


## Conferências
Contagens exatas, mediana com tolerância de 0,000001 hora. Também comparamos fila e desconhecidos com o resumo da etapa 06 e testamos ausência, futuro, corte exato, desconhecido e população vazia.

In [4]:
analise['testar_limites']()
print('Casos controlados: passaram.')
print('Combinações SQL/Python conferidas:', analise['conferir'](resultados))

Casos controlados: passaram.
Combinações SQL/Python conferidas: 710


## Próximos passos
Para um cartão de período, a referência é a última data elegível selecionada, mesmo se a fila estiver vazia. Não usar a média das medianas nem voltar silenciosamente ao último dia preenchido.

Volume e idade devem ser lidos juntos: 1.869 chamados em 16/03 têm mediana de 157,45 h; 33 em 15/12 têm mediana de 5.365,77 h. São composições diferentes do histórico disponível. A segunda medida foi conferida contra o chamado central da fotografia.

Agora podemos preparar o calendário e os relacionamentos do Power BI.

Validação: células executadas sequencialmente no Python do projeto, com saídas e estrutura JSON conferidas. Motor Jupyter não executado; suas dependências não estão instaladas.